In [1]:
from hospital_analysis import (
    charger_donnees,
    histogramme,
    camembert,
    barres,
    boite,
    nuage_points,
    interp_maladies,
    interp_box_departement,
    interp_duree_moyenne,
    render_interpretation
)
from IPython.display import display, HTML

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report



<div style="text-align: center; padding: 25px; background-color: #f8f9fa; border-radius: 12px; border-left: 8px solid #2c3e50; box-shadow: 2px 2px 10px rgba(0,0,0,0.1);">
    <h1 style="color: #2c3e50; font-family: 'Helvetica Neue', Arial, sans-serif; font-weight: bold; margin-bottom: 10px;">
        TP : Analyse de Données Hospitalières
    </h1>
    <h2 style="color: #34495e; font-family: 'Helvetica Neue', Arial, sans-serif; margin-top: 0; font-weight: 300;">
        Data Visualisation & Machine Learning
    </h2>
    <hr style="width: 50%; border: 1px solid #bdc3c7;">
    <p style="color: #223233; font-style: italic; font-size: 15px;">
        Exploration de l'activité des services, prédiction des coûts et classification des durées d'hospitalisation
    </p>
</div>

<div style="margin-top: 15px; padding: 18px; background-color: #f8f9fa; border-radius: 10px; border-left: 6px solid #2c3e50; color:#2c3e50;">
    <p><b>Fatou Bintou Gueye</b></p>
     <p><b>uicode.byfatoubintg@gmail.com</b> </p>
    <p><b>MBA-CDSD</b></p>
    <p><b>2025-2026</b> </p>
</div>

### **Analyse exploratoire et visualisation**

### **Partie 1 : Exploration des données**

1.	**Décrire la structure de la base de données.**

In [2]:
df = charger_donnees("/Users/fabynuur/Desktop/DataVis1/DataViz-hospital/hospital_data.csv")

html = f"""
<p>La base de données contient des informations sur les patients à savoir leur âge, leur sexe, leur maladie, le département auquel ils sont affectés, la date d'admission, la durée du séjour, la date de sortie, leur traitement et le coût.</p>
<p>Voici un aperçu des 5 premières lignes de la base de données :</p>
{df.head().to_html(index=False)}
"""
display(HTML(html))

PatientID,Age,Sexe,Departement,Maladie,DureeSejour,Cout,DateAdmission,DateSortie,Traitement
1,82,M,Cardiologie,Cancer,5,2125,03/09/2025,08/09/2025,Chirurgie
2,18,M,Oncologie,Cancer,15,8685,16/01/2025,31/01/2025,Médication
3,76,F,Cardiologie,Infarctus,2,822,17/05/2025,19/05/2025,Chirurgie
4,65,M,Neurologie,Fracture,12,7584,08/06/2025,20/06/2025,Physiothérapie
5,70,F,Orthopédie,Alzheimer,10,4420,01/10/2025,11/10/2025,Médication


In [3]:
#display(df.describe())

2.	**Visualiser la distribution de l’âge des patients.**

In [4]:
histogramme(
    df, x="Age",
    titre="Distribution de l'âge des patients",
    labels={"Age": "Âge du patient"},
    couleur="#df8116",
    titre_y="Nombre de patients",
).show()

In [5]:
age_moyen = df["Age"].mean()
age_min = df["Age"].min()
age_max = df["Age"].max()

render_interpretation(
    [
        (
            "L'hôpital accueille une population très diversifiée en termes d'âge. "
            f"L'âge moyen des patients est de <b>{age_moyen:.1f}</b> ans, "
            f"avec des extrêmes allant de <b>{age_min}</b> à <b>{age_max}</b> ans.\n"
            "\n Il y'a quand meme plus de patients dans la tranche d'âge de 55 à 60 ans (38)."
        )
    ],
    title="Interprétation de la distribution d'âge",
)

3.	**Analyser la répartition des patients selon le sexe.**

In [6]:
camembert(
    df, noms="Sexe",
    titre="Répartition des patients selon le sexe",
    couleurs={"M": "#125481", "F": "#ec7f26"},
).show()

In [7]:
repartition_sexe = df["Sexe"].value_counts(normalize=True) * 100
pct_h = repartition_sexe.get("M", 0)
pct_f = repartition_sexe.get("F", 0)

render_interpretation(
    [
        (
            "La répartition des patients selon le sexe est globalement équilibrée. "
            f"On observe environ <b>{pct_h:.1f} %</b> d'hommes et <b>{pct_f:.1f} %</b> de femmes."
        )
    ],
    title="Interprétation de la répartition par sexe",
)

4.	**Identifier les départements recevant le plus de patients.**

In [8]:
data_dept = df["Departement"].value_counts().reset_index(name="Nombre")

barres(
    data_dept, x="Departement", y="Nombre",
    titre="Nombre de patients par département",
    labels={"Nombre": "Nombre de patients", "Departement": "Département"},
    couleur="#125481",
    titre_x="Département", titre_y="Nombre de patients",
).show()

In [9]:
dept_plus_charge = data_dept.iloc[0]
dept_moins_charge = data_dept.iloc[-1]

total_patients = data_dept["Nombre"].sum()
part_plus_charge = (dept_plus_charge["Nombre"] / total_patients) * 100
part_top3 = (data_dept.head(3)["Nombre"].sum() / total_patients) * 100
ecart_dept_patients = dept_plus_charge["Nombre"] - dept_moins_charge["Nombre"]

render_interpretation(
    [
        (
            f"Le département <b>{dept_plus_charge['Departement']}</b> reçoit le plus de patients "
            f"(<b>{dept_plus_charge['Nombre']}</b>, soit <b>{part_plus_charge:.1f} %</b> du total), "
            f"tandis que <b>{dept_moins_charge['Departement']}</b> en reçoit le moins "
            f"(<b>{dept_moins_charge['Nombre']}</b>). "
            f"L'écart entre ces deux départements est de <b>{ecart_dept_patients}</b> patients, "
            f"ce qui indique une charge d'activité inégale. "
            f"Les 3 départements les plus fréquentés concentrent à eux seuls <b>{part_top3:.1f} %</b> des patients."
        )
    ],
    title="Interprétation du nombre de patients par département",
)

5.	**Identifier les maladies les plus fréquentes.**

In [10]:
data_maladie = df["Maladie"].value_counts().reset_index()
data_maladie.columns = ["Maladie", "Nombre de cas"]

barres(
    data_maladie, x="Nombre de cas", y="Maladie",
    orientation="h",
    titre="Maladies les plus fréquentes chez les patients",
    couleur_col="Nombre de cas", echelle_couleur="Blues",
    texte="Nombre de cas",
    tri_y="total ascending",
    titre_x="Nombre de cas", titre_y="Maladie",
).show()

In [11]:
interp_maladies(df)

### **Partie 2 : Analyse des hospitalisations**

6.	**Calculer la durée moyenne de séjour.**

In [12]:
interp_duree_moyenne(df)

7.	**Comparer la durée moyenne de séjour par département.**

In [13]:
boite(
    df, x="Departement", y="DureeSejour",
    titre="Répartition de la durée de séjour par département",
    labels={"DureeSejour": "Durée de séjour (jours)", "Departement": "Département"},
    couleur_col="Departement",
    titre_x="Département", titre_y="Durée de séjour (jours)",
).show()

In [14]:
interp_box_departement(df)

8.	**Identifier les maladies associées aux séjours les plus longs.**

In [15]:
data_duree_maladie = (
    df.groupby("Maladie")["DureeSejour"]
    .mean()
    .reset_index()
    .sort_values(by="DureeSejour", ascending=True)
)

barres(
    data_duree_maladie, x="DureeSejour", y="Maladie",
    orientation="h",
    titre="Durée moyenne de séjour selon la maladie",
    labels={"DureeSejour": "Durée moyenne (jours)", "Maladie": "Maladie"},
    couleur_col="DureeSejour", echelle_couleur="Oranges",
    texte=True,
).show()

In [16]:
interpretation = (
    "La maladie d'Alzheimer a les séjours les plus longs. "
    "La prise en charge des maladies comme cette dernière immobilise beaucoup les lits. "
    "Cela explique la saturation croisée avec les services de Neurologie et Gériatrie. "
    "L'hôpital doit anticiper des besoins à venir dans ces secteurs."
)

render_interpretation(
    [interpretation],
    title="Interprétation des séjours les plus longs",
)

9.	**Visualiser la répartition des traitements.**

In [17]:
import plotly.express as px

camembert(
    df, noms="Traitement",
    titre="Répartition des différents types de traitements",
    couleurs=px.colors.qualitative.Set2,
    position_texte="outside",
).show()

10.	**Analyser la relation entre âge et durée de séjour.**

In [18]:
nuage_points(
    df, x="Age", y="DureeSejour",
    titre="Relation entre l'âge et la durée de séjour",
    labels={"Age": "Âge", "DureeSejour": "Durée de séjour (jours)"},
    couleur_col="Maladie",
    titre_x="Âge du patient", titre_y="Durée de séjour (jours)",
).show()

### **Partie 3 : Analyse des coûts**

11.	**Calculer le coût moyen d’hospitalisation.**

In [19]:
cout_moyen = df["Cout"].mean()
cout_min = df["Cout"].min()
cout_max = df["Cout"].max()

interpretation = (
    "Le coût moyen d'une hospitalisation est un indicateur global du niveau de dépense. "
    "Ici, il permet d'estimer le budget type mobilisé pour un patient. "
    "L'écart entre le coût minimum et le coût maximum montre qu'il existe des prises en charge très différentes "
    "selon les cas (gravité, durée de séjour, traitement et service concerné)."
    "Il n'y a pas de 'patient standard' sur le plan financier, ce qui rend l'anticipation budgétaire complexe."
)

html = f"""
<div style='padding:12px;border-left:4px solid #125481;background:#f7fbff;border-radius:8px; color:#2c3e50;'>
  <h4 style='margin:0 0 8px 0;color:#125481;'>Interprétation du coût d'hospitalisation</h4>
  <p style='margin:4px 0;'><b>Coût moyen :</b> {cout_moyen:,.2f} €</p>
  <p style='margin:4px 0;'><b>Intervalle observé :</b> de {cout_min:,.2f} € à {cout_max:,.2f} €</p>
  <p style='margin:8px 0 0 0;line-height:1.5;'>{interpretation}</p>
</div>
"""
display(HTML(html))

12.	**Comparer les coûts par département.**

In [20]:
cout_par_dept = (
    df.groupby("Departement")["Cout"]
    .mean()
    .reset_index()
    .sort_values(by="Cout", ascending=False)
)

barres(
    cout_par_dept,
    x="Departement",
    y="Cout",
    titre="Coût moyen d'hospitalisation par département",
    labels={"Departement": "Département", "Cout": "Coût moyen (€)"},
    couleur_col="Cout",
    echelle_couleur="Blues",
    texte="Cout",
    titre_x="Département",
    titre_y="Coût moyen (€)",
).show()

dept_plus_cher = cout_par_dept.iloc[0]
dept_moins_cher = cout_par_dept.iloc[-1]
ecart_dept = dept_plus_cher["Cout"] - dept_moins_cher["Cout"]
ratio_dept = dept_plus_cher["Cout"] / dept_moins_cher["Cout"]

html = f"""
<div style='margin-top:10px;padding:10px;border-left:4px solid #125481;background:#f7fbff;border-radius:8px;color:#2c3e50;'>
  Le département <b>{dept_plus_cher['Departement']}</b> présente le coût moyen le plus élevé
  ({dept_plus_cher['Cout']:,.2f} €), tandis que <b>{dept_moins_cher['Departement']}</b> est le plus bas
  ({dept_moins_cher['Cout']:,.2f} €). L'écart est de <b>{ecart_dept:,.2f} €</b>, soit environ
  <b>{ratio_dept:.2f} fois</b> plus élevé entre les extrêmes. Cela montre les différences de besoins et ressources dans les départements.
</div>
"""
display(HTML(html))

13.	**Identifier les maladies générant les coûts les plus élevés.**

In [21]:
cout_par_maladie = (
    df.groupby("Maladie")["Cout"]
    .mean()
    .reset_index()
    .sort_values(by="Cout", ascending=True)
)

barres(
    cout_par_maladie,
    x="Cout",
    y="Maladie",
    orientation="h",
    titre="Coût moyen d'hospitalisation selon la maladie",
    labels={"Maladie": "Maladie", "Cout": "Coût moyen (€)"},
    couleur_col="Cout",
    echelle_couleur="Reds",
    texte="Cout",
    titre_x="Coût moyen (€)",
    titre_y="Maladie",
).show()

top5_maladies = cout_par_maladie.sort_values(by="Cout", ascending=False).head(5)
maladie_plus_chere = top5_maladies.iloc[0]
maladie_moins_chere = cout_par_maladie.iloc[0]
ecart_maladie = maladie_plus_chere["Cout"] - maladie_moins_chere["Cout"]

html = f"""
<div style='margin-top:10px;padding:10px;border-left:4px solid #b23a48;background:#fff6f6;border-radius:8px;color:#2c3e50;'>
La maladie <b>{maladie_plus_chere['Maladie']}</b> est associée au coût moyen le plus élevé
  ({maladie_plus_chere['Cout']:,.2f} €). À l'inverse, <b>{maladie_moins_chere['Maladie']}</b> présente le coût moyen le plus faible
  ({maladie_moins_chere['Cout']:,.2f} €).
</div>
"""
display(HTML(html))

14.	**Analyser la relation entre la durée de séjour et le coût.**

In [22]:
nuage_points(
    df,
    x="DureeSejour",
    y="Cout",
    titre="Relation entre la durée de séjour et le coût",
    labels={"DureeSejour": "Durée de séjour (jours)", "Cout": "Coût (€)"},
    couleur_col="Departement",
    titre_x="Durée de séjour (jours)",
    titre_y="Coût (€)",
).show()

corr_duree_cout = df[["DureeSejour", "Cout"]].corr().iloc[0, 1]

if corr_duree_cout >= 0.7:
    force_lien = "forte"
elif corr_duree_cout >= 0.4:
    force_lien = "modérée"
elif corr_duree_cout >= 0.2:
    force_lien = "faible"
elif corr_duree_cout > -0.2:
    force_lien = "très faible"
else:
    force_lien = "inverse"

html = f"""
<div style='padding:10px;border-left:4px solid #ec7f26;background:#fff8f2;border-radius:8px;color:#2c3e50;'>
  <b>Corrélation linéaire durée-coût :</b> {corr_duree_cout:.3f}<br>
  <b>Interprétation :</b> la relation entre la durée de séjour et le coût est <b>{force_lien}</b>. La Radiothérapie est le traitement le plus cher et a une durée de séjour relativement longue.
  Globalement, plus un patient reste longtemps hospitalisé, plus le coût tend à augmenter. La duréé d'occupation des lits donc est le principal facteur de coût bien plus que le type de traitement lui-même. Réduire la durée des séjours doit être la priorité numéro un pour optimiser le budget
</div>
"""
display(HTML(html))

15.	**Comparer les coûts selon les traitements.**

In [23]:
boite(
    df,
    x="Traitement",
    y="Cout",
    titre="Comparaison des coûts selon les traitements",
    labels={"Traitement": "Traitement", "Cout": "Coût (€)"},
    couleur_col="Traitement",
    titre_x="Traitement",
    titre_y="Coût (€)",
).show()

cout_par_traitement = (
    df.groupby("Traitement")["Cout"]
    .mean()
    .reset_index()
    .sort_values(by="Cout", ascending=False)
)

traitement_plus_cher = cout_par_traitement.iloc[0]
traitement_moins_cher = cout_par_traitement.iloc[-1]
ecart_traitement = traitement_plus_cher["Cout"] - traitement_moins_cher["Cout"]

html = f"""
<div style='margin-top:10px;padding:10px;border-left:4px solid #125481;background:#f7fbff;border-radius:8px;color:#2c3e50;'>
   Le traitement <b>{traitement_plus_cher['Traitement']}</b> est le plus coûteux en moyenne
  ({traitement_plus_cher['Cout']:,.2f} €), alors que <b>{traitement_moins_cher['Traitement']}</b> est le moins coûteux
  ({traitement_moins_cher['Cout']:,.2f} €). L'écart moyen est de <b>{ecart_traitement:,.2f} €</b>.
</div>
"""
display(HTML(html))

### **Partie Machine Learning**

L’objectif est de construire un modèle permettant d’expliquer ou de prédire un phénomène observé dans les données.

Et on va partir sur la prédiction du coût d'hospitalisation (régression)

**Préparation des données pour le machine learning et séparation en train/test.**

In [24]:
features_reg = ["Age", "Sexe", "Departement", "Maladie", "Traitement", "DureeSejour"]
target_reg = "Cout"

df_reg = df[features_reg + [target_reg]].copy()

df_reg = df_reg.dropna(subset=[target_reg])

In [25]:


X_reg = df_reg[features_reg]
y_reg = df_reg[target_reg]

num_features = ["Age", "DureeSejour"]
cat_features = ["Sexe", "Departement", "Maladie", "Traitement"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_reg = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Taille train (regression): {X_train_reg.shape}")
print(f"Taille test  (regression): {X_test_reg.shape}")

Taille train (regression): (400, 6)
Taille test  (regression): (100, 6)


**Construction de plusieurs modèles**

In [26]:
# Entrainement de plusieurs modeles de regression
reg_models = {
    "Regression lineaire": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

reg_results = []
reg_fitted_pipelines = {}

for name, model in reg_models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor_reg),
        ("model", model),
    ])
    pipe.fit(X_train_reg, y_train_reg)
    y_pred = pipe.predict(X_test_reg)

    mae = mean_absolute_error(y_test_reg, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    r2 = r2_score(y_test_reg, y_pred)

    reg_results.append({
        "Modele": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })
    reg_fitted_pipelines[name] = pipe

df_reg_results = pd.DataFrame(reg_results).sort_values(by="R2", ascending=False)
display(df_reg_results.style.format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"}))

,Modele,MAE,RMSE,R2
2,Gradient Boosting,774.10,980.77,0.830
0,Regression lineaire,792.10,1016.21,0.817
1,Random Forest,778.10,1041.14,0.808


In [27]:
best_reg_model_name = df_reg_results.iloc[0]["Modele"]
best_reg_pipe = reg_fitted_pipelines[best_reg_model_name]

print("Q4) Comparaison des performances des modeles (plus MAE/RMSE faibles et R2 eleve, meilleur est le modele):")
display(
    df_reg_results[["Modele", "MAE", "RMSE", "R2"]]
    .reset_index(drop=True)
    .style.format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"})
)
print(f" Le Meilleur modele selon R2: {best_reg_model_name}")

feature_names_reg = best_reg_pipe.named_steps["preprocess"].get_feature_names_out()
model_reg = best_reg_pipe.named_steps["model"]

if hasattr(model_reg, "feature_importances_"):
    importance_values = model_reg.feature_importances_
    importance_type = "Importance"
elif hasattr(model_reg, "coef_"):
    coef = model_reg.coef_
    importance_values = np.abs(coef)
    importance_type = "|Coefficient|"
else:
    importance_values = np.zeros(len(feature_names_reg))
    importance_type = "Importance"

importance_reg_df = pd.DataFrame({
    "Variable": feature_names_reg,
    importance_type: importance_values,
})

def variable_source(feature_name: str) -> str:
    if feature_name.startswith("num__Age"):
        return "Age"
    if feature_name.startswith("num__DureeSejour"):
        return "DureeSejour"
    if feature_name.startswith("cat__Sexe_"):
        return "Sexe"
    if feature_name.startswith("cat__Departement_"):
        return "Departement"
    if feature_name.startswith("cat__Maladie_"):
        return "Maladie"
    if feature_name.startswith("cat__Traitement_"):
        return "Traitement"
    return feature_name

importance_par_variable = (
    importance_reg_df.assign(VariableSource=importance_reg_df["Variable"].map(variable_source))
    .groupby("VariableSource", as_index=False)[importance_type]
    .sum()
    .sort_values(by=importance_type, ascending=False)
    .reset_index(drop=True)
)

print("\nQ5) Variables les plus importantes expliquant les couts (importance agregee):")
display(importance_par_variable.style.format({importance_type: "{:.4f}"}))

top3 = importance_par_variable.head(3)["VariableSource"].tolist()
print(f"Top 3 variables explicatives: {', '.join(top3)}")

render_interpretation(
    [
        f"Le meilleur modele de regression est <b>{best_reg_model_name}</b> (selection sur R2).",
        f"Variables les plus explicatives du cout : <b>{', '.join(top3)}</b>.",
    ],
    title="Synthese Q4-Q5",
    container_style="margin-top:14px; padding:16px 18px; border-left:6px solid #1f5a7a; background:linear-gradient(135deg,#f7fbff 0%,#eef6fb 100%); border-radius:12px; color:#10222f; box-shadow:0 6px 16px rgba(21,52,72,.10); font-family:'Segoe UI',Tahoma,sans-serif; line-height:1.55;",
    title_style="margin:0 0 10px 0; color:#174964; font-size:1.05rem; letter-spacing:.2px;",
    paragraph_style="margin:8px 0; color:#163447;",
)

Q4) Comparaison des performances des modeles (plus MAE/RMSE faibles et R2 eleve, meilleur est le modele):


,Modele,MAE,RMSE,R2
0,Gradient Boosting,774.10,980.77,0.830
1,Regression lineaire,792.10,1016.21,0.817
2,Random Forest,778.10,1041.14,0.808


 Le Meilleur modele selon R2: Gradient Boosting

Q5) Variables les plus importantes expliquant les couts (importance agregee):


,VariableSource,Importance
0,DureeSejour,0.9135
1,Age,0.0304
2,Traitement,0.0236
3,Maladie,0.0168
4,Departement,0.0130
5,Sexe,0.0027


Top 3 variables explicatives: DureeSejour, Age, Traitement


**Prédiction des séjours longs (classification)**

In [28]:
# Preparation des donnees de classification
df_clf = df.copy()
mean_stay = df_clf["DureeSejour"].mean()
df_clf["SejourLong"] = (df_clf["DureeSejour"] > mean_stay).astype(int)

features_clf = ["Age", "Sexe", "Departement", "Maladie", "Traitement", "Cout"]
target_clf = "SejourLong"

df_clf = df_clf[features_clf + [target_clf]].dropna(subset=[target_clf])
X_clf = df_clf[features_clf]
y_clf = df_clf[target_clf]

num_features_clf = ["Age", "Cout"]
cat_features_clf = ["Sexe", "Departement", "Maladie", "Traitement"]

numeric_transformer_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor_clf = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_clf, num_features_clf),
        ("cat", categorical_transformer_clf, cat_features_clf),
    ]
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"La moyenne de DureeSejour: {mean_stay:.2f} jours")
# print("Repartition SejourLong (0/1):")
# display(y_clf.value_counts(normalize=True).rename("proportion"))

# Entrainement des modeles de classification
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}

clf_results = []
clf_fitted_pipelines = {}

for name, model in clf_models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor_clf),
        ("model", model),
    ])
    pipe.fit(X_train_clf, y_train_clf)
    y_pred = pipe.predict(X_test_clf)

    clf_results.append({
        "Modele": name,
        "Accuracy": accuracy_score(y_test_clf, y_pred),
        "Precision": precision_score(y_test_clf, y_pred, zero_division=0),
        "Recall": recall_score(y_test_clf, y_pred, zero_division=0),
        "F1-score": f1_score(y_test_clf, y_pred, zero_division=0),
    })
    clf_fitted_pipelines[name] = pipe

df_clf_results = pd.DataFrame(clf_results).sort_values(by="F1-score", ascending=False)
display(df_clf_results.style.format({"Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}", "F1-score": "{:.3f}"}))

best_clf_model_name = df_clf_results.iloc[0]["Modele"]
best_clf_pipe = clf_fitted_pipelines[best_clf_model_name]
y_pred_best = best_clf_pipe.predict(X_test_clf)
print(f"\nMeilleur modele classification: {best_clf_model_name}")
print(classification_report(y_test_clf, y_pred_best))

# Facteurs associes aux sejours longs
feature_names_clf = best_clf_pipe.named_steps["preprocess"].get_feature_names_out()
model_clf = best_clf_pipe.named_steps["model"]

if hasattr(model_clf, "feature_importances_"):
    clf_importance = model_clf.feature_importances_
    clf_importance_type = "Importance"
elif hasattr(model_clf, "coef_"):
    clf_importance = np.abs(model_clf.coef_.ravel())
    clf_importance_type = "|Coefficient|"
else:
    clf_importance = np.zeros(len(feature_names_clf))
    clf_importance_type = "Importance"

importance_clf_df = pd.DataFrame({
    "Variable": feature_names_clf,
    clf_importance_type: clf_importance,
}).sort_values(by=clf_importance_type, ascending=False)

def nettoyer_prefixes(variable: str) -> str:
    return variable.replace("num__", "").replace("cat__", "")

importance_clf_affichage = importance_clf_df.copy()
importance_clf_affichage["Variable"] = importance_clf_affichage["Variable"].apply(nettoyer_prefixes)


top5_clf = importance_clf_affichage.head(5)["Variable"].tolist()
resume_clf = (
    f"Meilleur modele : <b>{best_clf_model_name}</b> "
    f"(Accuracy={df_clf_results.iloc[0]['Accuracy']:.3f}, "
    f"Precision={df_clf_results.iloc[0]['Precision']:.3f}, "
    f"Recall={df_clf_results.iloc[0]['Recall']:.3f}, "
    f"F1={df_clf_results.iloc[0]['F1-score']:.3f})."
)
details_clf = (
    "<b>Variables les plus associees aux sejours longs :</b><br>"
    + "<br>".join([f"- {v}" for v in top5_clf])
)

render_interpretation(
    [resume_clf, details_clf],
    title="Synthese classification",
    container_style="margin-top:14px; padding:16px 18px; border-left:6px solid #1f5a7a; background:linear-gradient(135deg,#f7fbff 0%,#eef6fb 100%); border-radius:12px; color:#10222f; box-shadow:0 6px 16px rgba(21,52,72,.10); font-family:'Segoe UI',Tahoma,sans-serif; line-height:1.55;",
    title_style="margin:0 0 10px 0; color:#174964; font-size:1.05rem; letter-spacing:.2px;",
    paragraph_style="margin:8px 0; color:#163447;",
)

La moyenne de DureeSejour: 7.95 jours


,Modele,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.900,0.938,0.865,0.900
1,Random Forest,0.900,0.938,0.865,0.900
2,Decision Tree,0.870,0.842,0.923,0.881



Meilleur modele classification: Logistic Regression
              precision    recall  f1-score   support

           0       0.87      0.94      0.90        48
           1       0.94      0.87      0.90        52

    accuracy                           0.90       100
   macro avg       0.90      0.90      0.90       100
weighted avg       0.90      0.90      0.90       100



### 6. Interprétation des résultats

In [29]:
# Reponses guidees aux 4 questions du sujet
def nettoyer_prefixes(variable: str) -> str:
    return variable.replace("num__", "").replace("cat__", "")

def variable_generique(variable: str) -> str:
    v = nettoyer_prefixes(variable)
    if "_" in v:
        return v.split("_", 1)[0]
    return v

best_reg_row = df_reg_results.iloc[0]
best_clf_row = df_clf_results.iloc[0]

# Importance agregee pour la regression
col_imp_reg = [c for c in importance_reg_df.columns if c != "Variable"][0]
reg_group = (
    importance_reg_df.assign(VariableGen=importance_reg_df["Variable"].map(variable_generique))
    .groupby("VariableGen", as_index=False)[col_imp_reg]
    .sum()
    .sort_values(by=col_imp_reg, ascending=False)
    .reset_index(drop=True)
)
top_reg = reg_group.head(3)["VariableGen"].tolist()

# Importance agregee pour la classification
col_imp_clf = [c for c in importance_clf_df.columns if c != "Variable"][0]
clf_group = (
    importance_clf_df.assign(VariableGen=importance_clf_df["Variable"].map(variable_generique))
    .groupby("VariableGen", as_index=False)[col_imp_clf]
    .sum()
    .sort_values(by=col_imp_clf, ascending=False)
    .reset_index(drop=True)
)
top_clf = clf_group.head(3)["VariableGen"].tolist()

# Quelques profils concrets (modalites) pour enrichir la reponse 2
top_modalites_clf = [nettoyer_prefixes(v) for v in importance_clf_df.head(3)["Variable"].tolist()]

# Fiabilite globale
r2 = float(best_reg_row["R2"])
f1 = float(best_clf_row["F1-score"])
if r2 >= 0.80 and f1 >= 0.85:
    niveau_fiabilite = "élevée"
elif r2 >= 0.70 and f1 >= 0.75:
    niveau_fiabilite = "correcte"
else:
    niveau_fiabilite = "à renforcer"

reponse_1 = (
    "<b>1) Quels facteurs expliquent les coûts hospitaliers ?</b><br>"
    f"Les facteurs les plus explicatifs sont principalement <b>{', '.join(top_reg)}</b>. "
    "La durée de séjour reste le déterminant dominant du coût."
 )

reponse_2 = (
    "<b>2) Quels profils de patients ont les séjours les plus longs ?</b><br>"
    f"Les variables les plus associées aux séjours longs sont <b>{', '.join(top_clf)}</b>. "
    f"Parmi les modalités marquantes observées : <b>{', '.join(top_modalites_clf)}</b>."
 )

reponse_3 = (
    "<b>3) Les modèles prédictifs sont-ils fiables ?</b><br>"
    f"La fiabilité est <b>{niveau_fiabilite}</b> : "
    f"régression ({best_reg_row['Modele']}) avec R²={r2:.3f}, "
    f"classification ({best_clf_row['Modele']}) avec F1={f1:.3f}."
 )

reponse_4 = (
    "<b>4) Comment l’hôpital pourrait-il optimiser ses ressources ?</b><br>"
    "Prioriser la réduction des durées de séjour évitables, anticiper les services/pathologies les plus coûteux, "
    "et ajuster les effectifs/lits selon les profils à risque identifiés par les modèles."
 )

html = f"""
<div style='margin-top:14px; padding:16px 18px; border-left:6px solid #1f5a7a; background:linear-gradient(135deg,#f7fbff 0%,#eef6fb 100%); border-radius:12px; color:#10222f; box-shadow:0 6px 16px rgba(21,52,72,.10); font-family:"Segoe UI",Tahoma,sans-serif; line-height:1.55;'>
  <h3 style='margin:0 0 10px 0; color:#174964; font-size:1.05rem; letter-spacing:.2px;'>Réponses - Synthèse analytique</h3>
  <div style='display:grid; grid-template-columns:1fr 1fr; gap:12px 16px; align-items:start;'>
    <div style='margin:0; color:#163447;'>{reponse_1}</div>
    <div style='margin:0; color:#163447;'>{reponse_3}</div>
    <div style='margin:0; color:#163447;'>{reponse_2}</div>
    <div style='margin:0; color:#163447;'>{reponse_4}</div>
  </div>
</div>
"""

display(HTML(html))

In [30]:
conclusion = (
    "L'analyse met en évidence un système hospitalier où la <b>durée de séjour</b> "
    "joue un rôle central dans les coûts, tandis que certains profils cliniques et certains départements "
    "sont davantage associés aux séjours longs. Les modèles construits présentent une fiabilité globalement "
    "satisfaisante pour orienter la décision opérationnelle."
)

recommandations = (
    "<b>Recommandations prioritaires :</b><br>"
    "1. Mettre en place un pilotage des durées de séjour par service (suivi hebdomadaire des cas longs).<br>"
    "2. Déployer un tri en amont des patients à risque de séjour long pour anticiper les besoins en lits.<br>"
    "3. Renforcer les ressources sur les pôles les plus sollicités (effectifs, planification, coordination).<br>"
    "4. Utiliser les prédictions de coûts pour améliorer la préparation budgétaire et la gestion des achats.<br>"
    "5. Actualiser périodiquement les modèles avec de nouvelles données pour maintenir leur performance."
)

render_interpretation(
    [conclusion, recommandations],
    title="Conclusion et recommandations",
    container_style="margin-top:14px; padding:16px 18px; border-left:6px solid #1f5a7a; background:linear-gradient(135deg,#f7fbff 0%,#eef6fb 100%); border-radius:12px; color:#10222f; box-shadow:0 6px 16px rgba(21,52,72,.10); font-family:'Segoe UI',Tahoma,sans-serif; line-height:1.55;",
    title_style="margin:0 0 10px 0; color:#174964; font-size:1.05rem; letter-spacing:.2px;",
    paragraph_style="margin:8px 0; color:#163447;",
)